In [1]:
from langgraph.graph import StateGraph, START, END
from langchain_google_genai import ChatGoogleGenerativeAI
from typing import TypedDict
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [3]:
model = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash-lite"
)

In [18]:
class BlogState(TypedDict):
    title: str
    outline: str
    content: str
    evaluation_score: int

In [20]:
def create_outline(state: BlogState) -> BlogState:
    title = state['title']
    
    prompt = f'Generate a detailed outline for a blog on the topic - {title}'
    outline = model.invoke(prompt).text

    state['outline'] = outline

    return state

In [21]:
def create_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']

    prompt = f'Write a detailed blog on the title - {title} using the follwing outline \n {outline}'

    content = model.invoke(prompt).text

    state['content'] = content

    return state

In [22]:
def evaluate_blog(state: BlogState) -> BlogState:
    title = state['title']
    outline = state['outline']
    content = state['content']

    prompt = f'Evaluate this blog on a scale of 1 to 5. Return only an integer from 1 to 5. where 1 is poor and 5 is excellent. Provide only the numeric score without any additional text.\n\nTitle: {title}\nOutline: {outline}\nContent: {content}'

    evaluation_score = model.invoke(prompt).text

    state['evaluation_score'] = int(evaluation_score.strip())

    return state

In [23]:
graph = StateGraph(BlogState)

graph.add_node('create_outline', create_outline)
graph.add_node('create_blog', create_blog)
graph.add_node('evaluate_blog', evaluate_blog)

graph.add_edge(START, 'create_outline')
graph.add_edge('create_outline', 'create_blog')
graph.add_edge('create_blog', 'evaluate_blog')
graph.add_edge('evaluate_blog', END)

workflow = graph.compile()

In [24]:
initial_state = {'title': 'Rise of AI in India'}
final_state = workflow.invoke(initial_state)

print(final_state['title'])
print("-" * 50)
print(final_state['outline'])
print("-" * 50)
print(final_state['content'])
print("-" * 50)
print(f"Evaluation Score: {final_state['evaluation_score']}")

Rise of AI in India
--------------------------------------------------
Here is a comprehensive and detailed outline for a blog post on the **"Rise of AI in India."** 

---

# Blog Title Options:
*   *Namaste AI: How India is Coding the Future of Artificial Intelligence*
*   *From Jugaad to Generative AI: The Unstoppable Rise of Artificial Intelligence in India*
*   *The Bharat AI Revolution: Opportunities, Challenges, and the Road Ahead*

---

## I. Introduction
*   **The Hook:** Start with a relatable scenario—e.g., asking a voice assistant in Hindi for local directions, or a farmer using an AI app to detect crop disease. 
*   **The Big Picture:** India is no longer just a back-office IT hub; it is rapidly transforming into a global AI powerhouse.
*   **Thesis Statement:** Driven by a massive talent pool, government initiatives, and unique societal challenges, India’s AI revolution is reshaping its economy, healthcare, agriculture, and daily life.

## II. The Catalyst: Why Now? (Key G